# LogCL + Path Head — chạy trên Google Colab

Chạy **3 dataset** (ICEWS14, ICEWS18, ICEWS05-15) × **2 cấu hình** (baseline, mức 2) × **3 seed** (123, 42, 2023) = **18 lần train**.

### ⚠️ Đọc trước khi chạy
- Colab free dùng **Python 3.12 + CUDA 12.x** → **không** cài thẳng được `torch 1.12 / dgl 1.0.2`. Notebook này tạo **env conda Python 3.9** đúng theo README.
- **18 lần train là RẤT nhiều** cho Colab free (giới hạn ~12h/phiên, hay bị ngắt). Notebook có cơ chế **bỏ qua run đã xong** (resumable) và **lưu kết quả ra Google Drive**. Nên chạy từng phần.
- **Khuyến nghị:** lần đầu để `DATASETS = ["ICEWS14"]` cho chạy thử 1 dataset, xác nhận pipeline OK rồi mới mở rộng.
- Chạy các cell **theo thứ tự**. Cell `condacolab` sẽ **tự restart runtime** — đó là bình thường; sau khi restart, chạy tiếp từ cell **ngay sau** nó (đừng chạy lại condacolab).
- Nhớ bật GPU: **Runtime > Change runtime type > T4 GPU**.

## Bước 0 — Kiểm tra môi trường

In [ ]:
!python --version
!nvidia-smi || echo "⚠️ Chưa bật GPU: Runtime > Change runtime type > GPU" 

## Bước 1 — Cài conda (condacolab)
Cell này sẽ **restart runtime**. Sau khi restart xong, chạy tiếp từ Bước 2.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## Bước 2 — Mount Google Drive (để lưu kết quả qua nhiều phiên)
Kết quả (file rank) lưu vào Drive nên không mất khi Colab ngắt → chạy lại notebook sẽ **tiếp tục** chỗ dở.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST_DIR = '/content/drive/MyDrive/logcl_out'   # nơi lưu rank
os.makedirs(PERSIST_DIR, exist_ok=True)
print('Lưu kết quả tại:', PERSIST_DIR)

## Bước 3 — Tạo env Python 3.9 + cài thư viện (~5–10 phút)

In [ ]:
%%bash
set -e
conda create -n logcl python=3.9 -y
conda run -n logcl pip install torch==1.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
conda install -n logcl -c dglteam/label/cu113 dgl=1.0.2 -y
conda install -n logcl -c conda-forge cudatoolkit=11.3 -y
conda run -n logcl pip install "numpy<2" tqdm pandas rdflib
echo "=== kiểm tra ==="
LD_LIBRARY_PATH=/usr/local/envs/logcl/lib:$LD_LIBRARY_PATH /usr/local/envs/logcl/bin/python -c \
  "import torch, dgl; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| dgl', dgl.__version__)" 

## Bước 4 — Clone repo + giải nén dữ liệu
👉 **Đổi `REPO_URL`** cho đúng repo bạn đã push.

In [ ]:
%%bash
set -e
cd /content
REPO_URL="https://github.com/buitrongtrinh/LogCL-PathHead.git"   # <-- ĐỔI nếu tên repo khác
rm -rf LogCL-PathHead
git clone "$REPO_URL"
cd LogCL-PathHead
unzip -q -o data.zip          # giải nén ra data/ (3 dataset + script xử lý)
echo "=== dataset có trong data/ ==="
ls data

## Bước 5 — Xử lý dữ liệu (static graph + history subgraph)
Tạo `e-w-graph.txt` (cho `--add-static-graph`) và các thư mục `his_dict/`, `his_graph_for/`, `his_graph_inv/`.
**Khá lâu**, nhất là ICEWS05-15. Chỉ cần chạy **một lần** (kết quả nằm trong `data/`, mất khi ngắt phiên → nếu ngắt phải chạy lại).

In [ ]:
%%bash
set -e
cd /content/LogCL-PathHead/data
ENVPY=/usr/local/envs/logcl/bin/python
export LD_LIBRARY_PATH=/usr/local/envs/logcl/lib:$LD_LIBRARY_PATH
# 1) static graph cho từng dataset
for D in ICEWS14 ICEWS18 ICEWS05-15; do
  echo "=== ent2word: $D ==="
  ( cd "$D" && $ENVPY ent2word.py )
done
# 2) history subgraph (get_his_subg.py tự lặp qua cả 3 dataset)
echo "=== get_his_subg (3 datasets) — có thể mất nhiều phút ==="
$ENVPY get_his_subg.py
echo "=== xong xử lý dữ liệu ==="


## Bước 6 — Cấu hình thí nghiệm
Chỉnh `DATASETS`, `SEEDS`, `CONFIGS` ở đây. Lần đầu nên để 1 dataset cho nhanh.

In [ ]:
# === CẤU HÌNH ===
DATASETS = ["ICEWS14", "ICEWS18", "ICEWS05-15"]   # bỏ bớt để chạy ít hơn, vd ["ICEWS14"]
SEEDS    = [123, 42, 2023]
CONFIGS  = ["base", "m2"]                           # base = không Path Head; m2 = Path Head level 2

# Siêu tham số mỗi dataset.
# ⚠️ ICEWS14 theo README; ICEWS18 & ICEWS05-15 là MẶC ĐỊNH HỢP LÝ — chỉnh train_history_len theo paper nếu cần.
HP = {
    "ICEWS14":    {"train_history_len": 7,  "n_hidden": 200, "lr": 0.001},
    "ICEWS18":    {"train_history_len": 9,  "n_hidden": 200, "lr": 0.001},
    "ICEWS05-15": {"train_history_len": 15, "n_hidden": 200, "lr": 0.001},
}

import os
OUT = os.path.join(globals().get("PERSIST_DIR", "/content/LogCL-PathHead"), "ranks")  # gom rank vào 1 thư mục
os.makedirs(OUT, exist_ok=True)
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/logcl/lib:" + os.environ.get("LD_LIBRARY_PATH", "")  # nối thêm, GIỮ driver path của Colab
print("Số lần train:", len(DATASETS)*len(CONFIGS)*len(SEEDS), "| lưu rank tại:", OUT)

## Bước 7 — Chạy train (RESUMABLE)
Tự **bỏ qua** run đã có file rank → có thể chạy lại notebook qua nhiều phiên, nó tiếp tục chỗ dở.
Nếu Colab ngắt giữa chừng, chỉ cần chạy lại từ Bước 1 (cài lại env) rồi tới cell này.

In [ ]:
import os, subprocess, time

REPO  = "/content/LogCL-PathHead"
ENVPY = "/usr/local/envs/logcl/bin/python"

COMMON = ("--dilate-len 1 --n-layers 2 --evaluate-every 1 --gpu 0 --self-loop "
          "--decoder convtranse --encoder uvrgcn --layer-norm --weight 0.5 "
          "--entity-prediction --angle 10 --discount 1 --pre-weight 0.9 --pre-type all "
          "--add-static-graph --use-cl --temperature 0.03")
PATH_FLAGS = "--use-path --path-dim 32 --path-layers 2 --path-batch-size 16 --path-level 2"

def run(dataset, config, seed):
    hp   = HP[dataset]
    dump = os.path.join(OUT, f"ranks_{config}_{dataset}_seed{seed}.csv")
    if os.path.exists(dump):
        print(f"[BỎ QUA] đã có {os.path.basename(dump)}"); return
    extra = PATH_FLAGS if config == "m2" else ""
    cmd = (f"{ENVPY} src/main.py -d {dataset} "
           f"--train-history-len {hp['train_history_len']} --test-history-len {hp['train_history_len']} "
           f"--lr {hp['lr']} --n-hidden {hp['n_hidden']} {COMMON} {extra} "
           f"--seed {seed} --dump-ranks {dump}")
    print(f"\n===== {dataset} | {config} | seed {seed} =====")
    t0 = time.time()
    subprocess.run(cmd, shell=True, cwd=REPO, env=os.environ, check=True)
    print(f"  ✅ xong sau {(time.time()-t0)/60:.1f} phút -> {os.path.basename(dump)}")

for d in DATASETS:
    for c in CONFIGS:
        for s in SEEDS:
            run(d, c, s)
print("\n=== HOÀN TẤT (hoặc đã bỏ qua các run có sẵn) ===")

## Bước 8 — Phân tích kết quả
Ổn định theo seed (base & mức 2), McNemar base-vs-mức2, và tách lặp-lại/mới.

In [ ]:
import os, subprocess
ENVPY = "/usr/local/envs/logcl/bin/python"; REPO = "/content/LogCL-PathHead"

def have(*paths): return [p for p in paths if os.path.exists(p)]

for d in DATASETS:
    print("\n" + "#"*70 + f"\n#  {d}\n" + "#"*70)
    base = [os.path.join(OUT, f"ranks_base_{d}_seed{s}.csv") for s in SEEDS]
    m2   = [os.path.join(OUT, f"ranks_m2_{d}_seed{s}.csv")   for s in SEEDS]

    fb = have(*base)
    if len(fb) >= 2:
        print(f"\n--- ổn định BASE ({d}) ---")
        subprocess.run([ENVPY, "analysis/analyze_stability.py", "--files", *fb], cwd=REPO, env=os.environ)
    fm = have(*m2)
    if len(fm) >= 2:
        print(f"\n--- ổn định MỨC 2 ({d}) ---")
        subprocess.run([ENVPY, "analysis/analyze_stability.py", "--files", *fm], cwd=REPO, env=os.environ)

    b0 = os.path.join(OUT, f"ranks_base_{d}_seed{SEEDS[0]}.csv")
    m0 = os.path.join(OUT, f"ranks_m2_{d}_seed{SEEDS[0]}.csv")
    if os.path.exists(b0) and os.path.exists(m0):
        print(f"\n--- base vs mức 2 — McNemar ({d}, seed {SEEDS[0]}) ---")
        subprocess.run([ENVPY, "analysis/analyze_transition.py", "--base", b0, "--new", m0], cwd=REPO, env=os.environ)
        print(f"\n--- lặp-lại vs mới ({d}, seed {SEEDS[0]}) ---")
        subprocess.run([ENVPY, "analysis/analyze_seen_unseen.py", "-d", d, "--base", b0, "--new", m0], cwd=REPO, env=os.environ)